# Experimentos de Extraccion de Biomarcadores de Voz para Parkinson

Cuaderno preparado para Google Colab como evidencia experimental del trabajo de extraccion de los biomarcadores usados por el contrato clasico de Parkinson/Oxford:

- `MDVP:Fo(Hz)`, `MDVP:Fhi(Hz)`, `MDVP:Flo(Hz)`
- `MDVP:Jitter(%)`, `MDVP:Jitter(Abs)`, `MDVP:RAP`, `MDVP:PPQ`, `Jitter:DDP`
- `MDVP:Shimmer`, `MDVP:Shimmer(dB)`, `Shimmer:APQ3`, `Shimmer:APQ5`, `MDVP:APQ`, `Shimmer:DDA`
- `NHR`, `HNR`
- `RPDE`, `DFA`, `spread1`, `spread2`, `D2`, `PPE`

Nota metodologica: las metricas de pitch, jitter, shimmer y harmonicidad se extraen con Parselmouth/Praat. Las variables no lineales (`RPDE`, `DFA`, `D2`, `PPE`, `spread1`, `spread2`) se calculan aqui como prototipos experimentales para investigacion y deben validarse antes de usarse en inferencia productiva.

## 1. Instalacion de dependencias

Ejecuta esta celda en Colab. Si Colab pide reiniciar el entorno despues de instalar alguna dependencia, reinicia y vuelve a ejecutar desde aqui.

In [ ]:
!pip -q install praat-parselmouth librosa soundfile scipy pandas matplotlib nolds openpyxl

## 2. Imports y configuracion general

In [ ]:
import math
import warnings
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import nolds
import numpy as np
import pandas as pd
import parselmouth
import soundfile as sf
from parselmouth.praat import call
from scipy import signal, stats

warnings.filterwarnings("ignore")

TARGET_SR = 16000
PITCH_FLOOR_HZ = 75.0
PITCH_CEILING_HZ = 300.0
MIN_PERIOD_SECONDS = 0.0001
MAX_PERIOD_SECONDS = 0.02
MAX_PERIOD_FACTOR = 1.3
MAX_AMPLITUDE_FACTOR = 1.6

PARKINSON_FEATURE_ORDER = [
    "MDVP:Fo(Hz)", "MDVP:Fhi(Hz)", "MDVP:Flo(Hz)", "MDVP:Jitter(%)", "MDVP:Jitter(Abs)",
    "MDVP:RAP", "MDVP:PPQ", "Jitter:DDP", "MDVP:Shimmer", "MDVP:Shimmer(dB)",
    "Shimmer:APQ3", "Shimmer:APQ5", "MDVP:APQ", "Shimmer:DDA", "NHR", "HNR",
    "RPDE", "DFA", "spread1", "spread2", "D2", "PPE",
]

## 3. Cargar audio

Sube una grabacion de vocal sostenida, idealmente `/a/`, de 3 a 10 segundos, con poco ruido y sin saturacion. Formatos comunes como WAV, WEBM, MP3, M4A u OGG suelen funcionar en Colab.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    AUDIO_PATH = next(iter(uploaded.keys()))
except Exception:
    AUDIO_PATH = "sample.wav"  # Cambia esta ruta si ejecutas el notebook fuera de Colab.

print("Audio seleccionado:", AUDIO_PATH)

## 4. Preprocesamiento reproducible

Se normaliza a mono, 16 kHz, se recorta silencio y se guarda un WAV PCM interno. Este es el mismo principio arquitectonico usado en MedDiag2: procesar a partir de una representacion canonica.

In [ ]:
def preprocess_audio(path, target_sr=TARGET_SR, top_db=25):
    y, sr = librosa.load(path, sr=target_sr, mono=True)
    y_trimmed, trim_index = librosa.effects.trim(y, top_db=top_db)
    if y_trimmed.size == 0:
        y_trimmed = y
    peak = np.max(np.abs(y_trimmed)) if y_trimmed.size else 0.0
    if peak > 0:
        y_trimmed = 0.95 * y_trimmed / peak
    out_path = "normalized_voice.wav"
    sf.write(out_path, y_trimmed, target_sr, subtype="PCM_16")
    return y_trimmed.astype(np.float32), target_sr, out_path, trim_index

y, sr, NORMALIZED_WAV, trim_index = preprocess_audio(AUDIO_PATH)
duration = len(y) / sr
print(f"WAV normalizado: {NORMALIZED_WAV}")
print(f"Sample rate: {sr} Hz")
print(f"Duracion efectiva: {duration:.3f} s")
print(f"Muestras: {len(y)}")

In [ ]:
plt.figure(figsize=(12, 3))
librosa.display.waveshow(y, sr=sr)
plt.title("Forma de onda normalizada")
plt.xlabel("Tiempo (s)")
plt.ylabel("Amplitud")
plt.grid(alpha=0.25)
plt.show()

## 5. Control de calidad experimental

Antes de extraer biomarcadores se calculan indicadores de calidad. Esta compuerta no reemplaza una validacion clinica, pero ayuda a documentar si el audio era razonable para analisis.

In [ ]:
def audio_quality_report(y, sr):
    abs_y = np.abs(y)
    rms = float(np.sqrt(np.mean(y ** 2))) if y.size else 0.0
    peak = float(np.max(abs_y)) if y.size else 0.0
    clipping = float(np.mean(abs_y >= 0.98)) if y.size else 0.0
    silence_threshold = max(0.003, peak * 0.02)
    silence_ratio = float(np.mean(abs_y < silence_threshold)) if y.size else 1.0
    duration = float(len(y) / sr) if sr else 0.0
    noise_level = float(np.median(np.abs(np.diff(y)))) if y.size > 1 else 0.0
    quality_flags = []
    if duration < 0.5:
        quality_flags.append("duracion_muy_corta")
    if rms < 0.005:
        quality_flags.append("nivel_muy_bajo")
    if clipping > 0.05:
        quality_flags.append("clipping_excesivo")
    if silence_ratio > 0.80:
        quality_flags.append("demasiado_silencio")
    return {
        "duration_seconds": duration,
        "rms": rms,
        "peak_amplitude": peak,
        "clipping_ratio": clipping,
        "silence_ratio": silence_ratio,
        "noise_level_proxy": noise_level,
        "quality_flags": ", ".join(quality_flags) if quality_flags else "ok",
    }

qc = audio_quality_report(y, sr)
pd.DataFrame([qc]).T.rename(columns={0: "valor"})

## 6. Biomarcadores clinicos con Parselmouth/Praat

Esta celda extrae las variables clasicas de frecuencia fundamental, jitter, shimmer y harmonicidad usando llamadas de Praat via Parselmouth.

In [ ]:
def finite_or_nan(value):
    try:
        value = float(value)
        return value if np.isfinite(value) else np.nan
    except Exception:
        return np.nan

def extract_praat_biomarkers(wav_path):
    sound = parselmouth.Sound(wav_path)
    pitch = sound.to_pitch(pitch_floor=PITCH_FLOOR_HZ, pitch_ceiling=PITCH_CEILING_HZ)
    pitch_values = pitch.selected_array["frequency"]
    voiced_pitch = pitch_values[pitch_values > 0]
    if voiced_pitch.size == 0:
        raise ValueError("No se detectaron cuadros sonoros; revisa calidad del audio.")

    point_process = call(sound, "To PointProcess (periodic, cc)", PITCH_FLOOR_HZ, PITCH_CEILING_HZ)
    harmonicity = call(sound, "To Harmonicity (cc)", 0.01, PITCH_FLOOR_HZ, 0.1, 1.0)

    hnr = finite_or_nan(call(harmonicity, "Get mean", 0, 0))
    nhr = float(10 ** (-hnr / 10.0)) if np.isfinite(hnr) else np.nan

    features = {
        "MDVP:Fo(Hz)": finite_or_nan(np.mean(voiced_pitch)),
        "MDVP:Fhi(Hz)": finite_or_nan(np.max(voiced_pitch)),
        "MDVP:Flo(Hz)": finite_or_nan(np.min(voiced_pitch)),
        "MDVP:Jitter(%)": finite_or_nan(call(point_process, "Get jitter (local)", 0, 0, MIN_PERIOD_SECONDS, MAX_PERIOD_SECONDS, MAX_PERIOD_FACTOR)),
        "MDVP:Jitter(Abs)": finite_or_nan(call(point_process, "Get jitter (local, absolute)", 0, 0, MIN_PERIOD_SECONDS, MAX_PERIOD_SECONDS, MAX_PERIOD_FACTOR)),
        "MDVP:RAP": finite_or_nan(call(point_process, "Get jitter (rap)", 0, 0, MIN_PERIOD_SECONDS, MAX_PERIOD_SECONDS, MAX_PERIOD_FACTOR)),
        "MDVP:PPQ": finite_or_nan(call(point_process, "Get jitter (ppq5)", 0, 0, MIN_PERIOD_SECONDS, MAX_PERIOD_SECONDS, MAX_PERIOD_FACTOR)),
        "MDVP:Shimmer": finite_or_nan(call([sound, point_process], "Get shimmer (local)", 0, 0, MIN_PERIOD_SECONDS, MAX_PERIOD_SECONDS, MAX_PERIOD_FACTOR, MAX_AMPLITUDE_FACTOR)),
        "MDVP:Shimmer(dB)": finite_or_nan(call([sound, point_process], "Get shimmer (local_dB)", 0, 0, MIN_PERIOD_SECONDS, MAX_PERIOD_SECONDS, MAX_PERIOD_FACTOR, MAX_AMPLITUDE_FACTOR)),
        "Shimmer:APQ3": finite_or_nan(call([sound, point_process], "Get shimmer (apq3)", 0, 0, MIN_PERIOD_SECONDS, MAX_PERIOD_SECONDS, MAX_PERIOD_FACTOR, MAX_AMPLITUDE_FACTOR)),
        "Shimmer:APQ5": finite_or_nan(call([sound, point_process], "Get shimmer (apq5)", 0, 0, MIN_PERIOD_SECONDS, MAX_PERIOD_SECONDS, MAX_PERIOD_FACTOR, MAX_AMPLITUDE_FACTOR)),
        "MDVP:APQ": finite_or_nan(call([sound, point_process], "Get shimmer (apq11)", 0, 0, MIN_PERIOD_SECONDS, MAX_PERIOD_SECONDS, MAX_PERIOD_FACTOR, MAX_AMPLITUDE_FACTOR)),
        "NHR": nhr,
        "HNR": hnr,
    }
    features["Jitter:DDP"] = features["MDVP:RAP"] * 3.0 if np.isfinite(features["MDVP:RAP"]) else np.nan
    features["Shimmer:DDA"] = features["Shimmer:APQ3"] * 3.0 if np.isfinite(features["Shimmer:APQ3"]) else np.nan
    return features, voiced_pitch

praat_features, voiced_pitch = extract_praat_biomarkers(NORMALIZED_WAV)
pd.DataFrame([praat_features]).T.rename(columns={0: "valor"})

## 7. Serie de pitch para variables no lineales

Las variables no lineales se calculan sobre una serie derivada de pitch/periodos. Esta decision es experimental y debe quedar documentada junto con los parametros usados.

In [ ]:
pitch_series = np.asarray(voiced_pitch, dtype=np.float64)
pitch_series = pitch_series[np.isfinite(pitch_series)]
pitch_series = pitch_series[(pitch_series >= PITCH_FLOOR_HZ) & (pitch_series <= PITCH_CEILING_HZ)]

period_series = 1.0 / pitch_series if pitch_series.size else np.array([])

plt.figure(figsize=(12, 3))
plt.plot(pitch_series, linewidth=1.5)
plt.title("Serie F0 sonorizada usada para features no lineales")
plt.xlabel("Indice de cuadro sonoro")
plt.ylabel("F0 (Hz)")
plt.grid(alpha=0.25)
plt.show()

print("Muestras F0 sonorizadas:", pitch_series.size)

## 8. Implementacion experimental de RPDE, DFA, D2, PPE, spread1 y spread2

Estas funciones sirven para documentar experimentos. No garantizan equivalencia exacta con el dataset Oxford original ni con MDVP. La salida debe compararse contra referencias o usarse para reentrenar un modelo propio con el mismo extractor.

In [ ]:
def clean_series(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return x
    return x - np.mean(x)

def sample_entropy_proxy(x, m=2, r=None):
    x = clean_series(x)
    if x.size < m + 3:
        return np.nan
    if r is None:
        r = 0.2 * np.std(x)
    if r == 0:
        return 0.0
    def _phi(order):
        windows = np.array([x[i:i + order] for i in range(len(x) - order + 1)])
        count = 0
        total = 0
        for i in range(len(windows)):
            dist = np.max(np.abs(windows - windows[i]), axis=1)
            count += np.sum(dist <= r) - 1
            total += len(windows) - 1
        return count / total if total else np.nan
    phi_m = _phi(m)
    phi_m1 = _phi(m + 1)
    if not np.isfinite(phi_m) or not np.isfinite(phi_m1) or phi_m <= 0 or phi_m1 <= 0:
        return np.nan
    return float(-np.log(phi_m1 / phi_m))

def rpde_proxy(x, bins=32):
    x = clean_series(x)
    if x.size < 16:
        return np.nan
    hist, _ = np.histogram(x, bins=bins, density=False)
    p = hist / np.sum(hist)
    p = p[p > 0]
    entropy = -np.sum(p * np.log(p))
    return float(entropy / np.log(bins))

def dfa_feature(x):
    x = clean_series(x)
    if x.size < 32:
        return np.nan
    try:
        return float(nolds.dfa(x))
    except Exception:
        return np.nan

def d2_feature(x):
    x = clean_series(x)
    if x.size < 64:
        return np.nan
    try:
        emb_dim = 3 if x.size < 200 else 4
        return float(nolds.corr_dim(x, emb_dim=emb_dim))
    except Exception:
        return np.nan

def ppe_proxy(periods):
    periods = np.asarray(periods, dtype=np.float64)
    periods = periods[np.isfinite(periods)]
    if periods.size < 8:
        return np.nan
    log_period = np.log(periods)
    detrended = signal.detrend(log_period)
    return sample_entropy_proxy(detrended)

def spread_features_from_pitch(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size < 8:
        return np.nan, np.nan
    log_f0 = np.log(x)
    descriptors = np.array([
        np.mean(log_f0),
        np.std(log_f0),
        stats.skew(log_f0),
        stats.kurtosis(log_f0),
        np.percentile(log_f0, 5),
        np.percentile(log_f0, 95),
    ], dtype=np.float64)
    descriptors = np.nan_to_num(descriptors, nan=0.0, posinf=0.0, neginf=0.0)
    # Proyeccion deterministica experimental para dejar evidencia de dispersion.
    spread1 = float(-abs(descriptors[1]) - abs(descriptors[2]))
    spread2 = float(descriptors[5] - descriptors[4])
    return spread1, spread2

spread1, spread2 = spread_features_from_pitch(pitch_series)
nonlinear_features = {
    "RPDE": rpde_proxy(period_series),
    "DFA": dfa_feature(period_series),
    "spread1": spread1,
    "spread2": spread2,
    "D2": d2_feature(period_series),
    "PPE": ppe_proxy(period_series),
}

pd.DataFrame([nonlinear_features]).T.rename(columns={0: "valor_experimental"})

## 9. Vector completo de 22 biomarcadores

Esta tabla combina los biomarcadores extraidos con Parselmouth y los prototipos no lineales. La columna `origen` deja trazabilidad del metodo usado.

In [ ]:
all_features = {**praat_features, **nonlinear_features}
feature_rows = []
for name in PARKINSON_FEATURE_ORDER:
    value = all_features.get(name, np.nan)
    origin = "Parselmouth/Praat" if name in praat_features else "Experimental nonlinear prototype"
    feature_rows.append({
        "feature": name,
        "value": value,
        "is_finite": bool(np.isfinite(value)),
        "origin": origin,
    })

features_df = pd.DataFrame(feature_rows)
features_df

In [ ]:
missing_or_invalid = features_df.loc[~features_df["is_finite"], "feature"].tolist()
print("Features finitas:", int(features_df["is_finite"].sum()), "/", len(features_df))
print("Features no finitas:", missing_or_invalid if missing_or_invalid else "ninguna")

features_df.to_csv("biomarcadores_voz_parkinson_experimento.csv", index=False)
features_df.to_excel("biomarcadores_voz_parkinson_experimento.xlsx", index=False)
print("Archivos exportados: biomarcadores_voz_parkinson_experimento.csv y .xlsx")

## 10. Interpretacion para MedDiag2

- Las features con origen `Parselmouth/Praat` son las candidatas mas fuertes para el pipeline productivo.
- Las features con origen `Experimental nonlinear prototype` documentan el trabajo de investigacion, pero requieren validacion con dataset de referencia, pruebas de reproducibilidad y comparacion contra implementaciones publicadas.
- Si se entrena un modelo nuevo con estas features experimentales, debe entrenarse y servirse usando exactamente este mismo extractor y versionarlo.
- No se recomienda imputar `0.0` cuando una feature no se puede calcular; es preferible marcarla como no finita y bloquear inferencia o usar un modelo compatible con el subconjunto disponible.